In [ ]:
from pathlib import Path

In [ ]:
folder_path = Path('/content')

print("Files ending with .csv or .tsv:")
for file in folder_path.iterdir():
  if file.is_file() and (file.suffix == '.csv' or file.suffix == '.tsv'):
    with open(file, 'r' , encoding="latin1") as f:
      comp_content = f.read()

Files ending with .csv or .tsv:


In [ ]:
import pandas as pd
import io


df = pd.read_csv(io.StringIO(comp_content), sep='\t')


display(df.head())

,tweet_id,tweet_text,label
0,'451585608612208640',Guys northem chile really needs a support mess...,sympathy_and_emotional_support
1,'451500842395250688',RT @Euphorian54: #Rehearsal time w/ @debbiegib...,not_related_or_irrelevant
2,'451833189439246336',Happy B-Day! @AllRiseSilver https://t.co/jcfCk...,not_related_or_irrelevant
3,'451330668539031552',RT @kuroab_90: My heart goes out to the victim...,sympathy_and_emotional_support
4,'451489071404040192',"Chile Earthquake: 5 Dead, Several Seriously In...",injured_or_dead_people


In [2]:
import pickle as pkl

In [ ]:
pkl.dump(df, open('compiled_data.pkl', 'wb'))

In [ ]:
comp = pkl.load(open('compiled_data.pkl', 'rb'))

In [ ]:
comp.head()

,tweet_id,tweet_text,label
0,'451585608612208640',Guys northem chile really needs a support mess...,sympathy_and_emotional_support
1,'451500842395250688',RT @Euphorian54: #Rehearsal time w/ @debbiegib...,not_related_or_irrelevant
2,'451833189439246336',Happy B-Day! @AllRiseSilver https://t.co/jcfCk...,not_related_or_irrelevant
3,'451330668539031552',RT @kuroab_90: My heart goes out to the victim...,sympathy_and_emotional_support
4,'451489071404040192',"Chile Earthquake: 5 Dead, Several Seriously In...",injured_or_dead_people


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
comp['tweet_id'] = comp['tweet_id'].str.replace("'", "").astype(int)

In [ ]:
comp.head()
comp['label'].unique()

array(['sympathy_and_emotional_support', 'not_related_or_irrelevant',
       'injured_or_dead_people', 'infrastructure_and_utilities_damage',
       'caution_and_advice', 'other_useful_information',
       'donation_needs_or_offers_or_volunteering_services',
       'missing_trapped_or_found_people',
       'displaced_people_and_evacuations'], dtype=object)

In [ ]:
severity_map = {
    "injured_or_dead_people":4,
    "missing_trapped_or_found_people":4,
    "displaced_people_and_evacuations":3,
    "infrastructure_and_utilities_damage":3,
    "donation_needs_or_offers_or_volunteering_services":2,
    "caution_and_advice":2,
    "sympathy_and_emotional_support":1,
    "other_useful_information":1,
    "not_related_or_irrelevant":0
}

1- Injured or dead people---Reports of casualties and/or injured people due to the crisis

2- Missing, trapped, or found people---Reports and/or questions about missing or found people

3- Displaced people and evacuations---People who have relocated due to the crisis, even for a short time (includes evacuations)

4- Infrastructure and utilities damage---Reports of damaged buildings, roads, bridges, or utilities/services interrupted or restored

5- Donation needs or offers or volunteering services---Reports of urgent needs or donations of shelter and/or supplies such as food, water, clothing, money, medical supplies or blood; and volunteering services

6- Caution and advice---Reports of warnings issued or lifted, guidance and tips

7- Sympathy and emotional support---Prayers, thoughts, and emotional support

8- Other useful information---Other useful information that helps understand
the situation

9- Not related or irrelevant---Unrelated to the situation or irrelevant


In [ ]:
comp['severity_class'] = comp['label'].map(severity_map)

In [ ]:
comp.head()

,tweet_id,tweet_text,label,severity_class
0,451585608612208640,Guys northem chile really needs a support mess...,sympathy_and_emotional_support,1
1,451500842395250688,RT @Euphorian54: #Rehearsal time w/ @debbiegib...,not_related_or_irrelevant,0
2,451833189439246336,Happy B-Day! @AllRiseSilver https://t.co/jcfCk...,not_related_or_irrelevant,0
3,451330668539031552,RT @kuroab_90: My heart goes out to the victim...,sympathy_and_emotional_support,1
4,451489071404040192,"Chile Earthquake: 5 Dead, Several Seriously In...",injured_or_dead_people,4


In [ ]:
pkl.dump(comp, open('compiled.pkl' , 'wb'))

In [1]:
import pickle as pkl
import pandas as pd
import numpy as np
import tensorflow as tf

In [3]:
emb = pkl.load(open('/content/embedding_matrix.pkl' , 'rb'))


In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint

In [5]:
def build_model(vocab_size, embed_dim, embedding_matrix, max_len=100 , num_classes=5):

    model = Sequential([
        Embedding(
            input_dim = vocab_size,
            output_dim = embed_dim,
            weights = [embedding_matrix],
            trainable = False,
            input_shape = (max_len,),
            name="embedding_layer"
        ),

        Bidirectional(LSTM(256, return_sequences=True) , name = "bilstm1"),
        Dropout(0.5),
        Bidirectional(LSTM(128 , return_sequences=False)),
        Dropout(0.5),
        Dense(64, activation="relu" , name="dense1"),
        Dense(num_classes, activation="softmax" , name="output")
    ])

    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [6]:
tf.keras.backend.clear_session()
model = build_model(4188, 300, emb , 100, num_classes=5)
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ (None, 100, 300)       │     1,256,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm1 (Bidirectional)         │ (None, 100, 512)       │     1,140,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       656,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,070,293 (11.71 MB)

 Trainable params: 1,813,893 (6.92 MB)

 Non-trainable params: 1,256,400 (4.79 MB)

In [7]:
x,y  = pkl.load(open('/content/X_y_sequences.pkl' , 'rb'))

In [8]:
x.shape , y.shape

((1932, 100), (1932,))

In [9]:

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [10]:
x_train , x_test, y_train, y_test = train_test_split(x,y, test_size=0.25, random_state=22)


In [11]:
x_train.shape , x_test.shape, y_train.shape, y_test.shape

((1449, 100), (483, 100), (1449,), (483,))

In [19]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
es = EarlyStopping(monitor = 'val_loss', patience=3)
mcp = ModelCheckpoint('best_model.weights.h5', monitor='val_loss', save_best_only=True, save_weights_only=True)

In [20]:
history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    verbose=1,
    callbacks = [mcp]
)

Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.8939 - loss: 0.3170 - val_accuracy: 0.7655 - val_loss: 0.7543
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9189 - loss: 0.2341 - val_accuracy: 0.6966 - val_loss: 0.9152
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.8991 - loss: 0.2911 - val_accuracy: 0.7448 - val_loss: 0.7371
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.9223 - loss: 0.2156 - val_accuracy: 0.7621 - val_loss: 0.9277
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9336 - loss: 0.2012 - val_accuracy: 0.7310 - val_loss: 0.9104
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9491 - loss: 0.1557 - val_accuracy: 0.7207 - val_loss: 0.9209
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9646 - loss: 0.1070 - val_accuracy: 0.7345 - val_loss: 1.1014
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9612 - loss: 0.1029 - val_accuracy: 0.7414 - v

In [28]:
from sklearn.metrics import classification_report , accuracy_score, roc_auc_score

In [23]:
y_pred = model.predict(x_test)
classification_report(np.argmax(y_pred, axis=1), y_test)


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step


'              precision    recall  f1-score   support\n\n           0       0.70      0.85      0.76        84\n           1       0.84      0.76      0.80       284\n           2       0.48      0.55      0.51        66\n           3       0.54      0.65      0.59        23\n           4       0.86      0.73      0.79        26\n\n    accuracy                           0.74       483\n   macro avg       0.68      0.71      0.69       483\nweighted avg       0.75      0.74      0.74       483\n'

In [26]:
accuracy_score(np.argmax(y_pred, axis=1), y_test)

0.7370600414078675

In [30]:
roc_auc_score(y_test , y_pred , multi_class='ovr', average='weighted')

np.float64(0.8762569622154289)